# 📝 LangChain 기본 구조 과제 LV3(통합) — 상담 봇·상품 설명 파이프라인

> 지금까지 배운 부품을 모아 **작은 프로그램 두 개**를 완성합니다. 각 문제는 **단계 셀**로 나뉩니다 — 각 단계의 요구사항을 그 셀에서 바로 확인할 수 있습니다.

## 풀이 방법
1. 맨 위 **준비 셀**을 먼저 실행하세요(`.env` 의 본인 `OPENAI_API_KEY` 가 필요합니다).
2. 각 단계 답안 셀을 채우고 자가채점 셀로 확인하세요. **앞 단계에서 만든 변수**를 뒤 단계가 씁니다(순서대로 실행).
3. 모델이 만든 문장은 실행할 때마다 다릅니다 — 채점은 **타입·구조**로 합니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] (1/2) 이 과제에서 공통으로 쓰는 부품들 — 실행하면 준비 끝입니다.
import os
import csv

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
parser = StrOutputParser()
print('부품 준비 완료 —', type(model).__name__, '+', type(parser).__name__)

In [ ]:
# [제공 코드] (2/2) LV2 에서 직접 만든 기록 관리 함수 — 여기서는 그대로 제공합니다.
#   이번 과제의 초점은 '이 부품들을 조립해 프로그램을 만드는 것' 이라, 이미 만든 것은 다시 안 만듭니다.
def add_turn(history, user_text, ai_text):
    """기록 뒤에 사용자 질문과 모델 답을 한 턴으로 붙인 새 리스트를 돌려준다."""
    return history + [('user', user_text), ('assistant', ai_text)]


def keep_recent(history, n_turns):
    """기록에서 최근 n_turns 턴(2*n_turns 줄)만 남긴다."""
    return history[-2 * n_turns:]


# 잘 불러졌는지 여기서 바로 확인합니다(한 턴 = 두 줄).
print(add_turn([], '안녕', '반갑습니다'))
print(keep_recent([('user', 'a'), ('assistant', 'b'), ('user', 'c'), ('assistant', 'd')], 1))

---
## 1. 중고거래 상담 봇
**배경**: 사용자의 입력을 **정리**하고, 앞 대화를 **기억**하며, 기록을 **최근 2턴**으로 유지하는 상담 봇을 단계별로 완성합니다.

> 6개 단계를 순서대로 진행합니다: (1) 입력 정리 부품·상담 체인 → (2) 상담 함수 `ask` → (3) 첫 질문 → (4) 이어 묻기(단기 기억) → (5) 장기 기억 확인 → (6) `while` 멀티턴.

> 이 상담 봇은 **단기 기억**(최근 대화)과 **장기 기억**(오래 쓸 사실)을 함께 씁니다. 장기 기억함은 교안 6절과 같은 도구라 아래 **제공 셀**로 드립니다 — 여러분은 그것을 **상담 체인·함수 안에서 쓰기만** 하면 됩니다.

In [ ]:
# [제공 코드] 장기 기억함 — 교안_02 6절과 같은 구성입니다(모델을 불러오는 데 잠시 걸립니다).
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
OUT_DIR = Path('output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
chroma = chromadb.PersistentClient(path=str(OUT_DIR / 'chroma_memory'))
if 'lv3_memory' in [c if isinstance(c, str) else c.name for c in chroma.list_collections()]:
    chroma.delete_collection('lv3_memory')
memory_box = chroma.get_or_create_collection('lv3_memory', metadata={'hnsw:space': 'cosine'})

def remember(fact):
    """오래 쓸 사실 한 줄을 장기 기억에 저장한다."""
    vector = embed_model.encode([fact], normalize_embeddings=True)
    memory_box.add(ids=[f'mem{memory_box.count()}'], documents=[fact], embeddings=vector.tolist())

def recall(question, k=2):
    """질문과 의미가 가까운 기억 k개를 문장 리스트로 돌려준다."""
    q_vec = embed_model.encode([question], normalize_embeddings=True)
    res = memory_box.query(query_embeddings=q_vec.tolist(), n_results=k)
    return res['documents'][0]

# 이전 상담에서 알아 둔 사실들 — 오늘 대화에는 나오지 않습니다.
for f in ['이 손님의 예산은 5만원 이하이고 정품 부속이 갖춰진 매물만 원한다.', '이 손님은 강남역 근처에서만 직거래가 가능하다.']:
    remember(f)
print('장기 기억 준비 완료 / 담긴 사실 수:', memory_box.count())

### 1단계 — 입력 정리 부품과 상담 체인 만들기
- 사용자 입력의 앞뒤·연속 공백을 한 칸으로 정리하는 함수 **`clean_input(text)`** 를 만들고 `RunnableLambda` 로 감싼 부품 **`clean_step`** 을 만드세요(`' '.join(text.split())`).
- 프롬프트를 `[('system', 역할 + 장기 기억 자리), MessagesPlaceholder('history'), ('human', '{input}')]` 로 만들고 `model | parser` 를 이어 체인 **`counsel_chain`** 을 만드세요. system 문구는 `'너는 중고 거래 플랫폼의 친절한 상담원이다. 존댓말로 간결하게 답한다.\n[기억해 둔 사실]\n{memory}'` 로 두어 **`{memory}` 변수**를 만드세요(교안 7절과 같은 구조).

**예시**: `clean_input('  a   b ')` → `'a b'`.

<details><summary>힌트</summary>

```text
접근방법:
- 공백 정리 함수를 만들고 RunnableLambda 로 감싼다. 상담 체인은 MessagesPlaceholder 를 넣어 만든다.

세부구현:
1. clean_input 은 ' '.join(text.split()) 를 반환한다.
2. clean_step = RunnableLambda(clean_input).
3. counsel_chain = (system(안에 {memory} 포함) + MessagesPlaceholder('history') + human) 프롬프트 | model | parser.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert clean_input('  a   b ') == 'a b'
assert clean_step.invoke('  x   y ') == 'x y'
assert set(counsel_chain.steps[0].input_variables) == {'memory', 'history', 'input'}
print('✅ 통과!')

### 2단계 — 상담 함수 `ask` 만들기
- 전역 기록 리스트 **`chat_log`**(빈 리스트로 시작)를 두고, 함수 **`ask(user_text)`** 를 만드세요. 동작 순서:
  1. **`clean_step`**(1단계에서 만든 부품)으로 입력을 정리한다 — 감싸 둔 부품을 실제로 쓰는 자리입니다.
  2. `recall(정리된입력)` 로 장기 기억에서 관련 사실을 꺼내 **한 덩어리 글**로 만든다(예: `'\n'.join(f'- {f}' for f in facts)`).
  3. `counsel_chain.invoke({'memory': 그 글, 'history': chat_log, 'input': 정리된입력})` 로 답을 받는다.
  4. `add_turn` 으로 (정리된 입력, 답)을 `chat_log` 에 누적한다.
  5. `keep_recent(chat_log, 10)` 로 최근 **10턴**만 남긴다(대화가 길어져도 프롬프트가 무한정 커지지 않게).
  6. 답을 반환한다.

**예시**: `ask` 를 한 번 부르면 `chat_log` 길이는 2(한 턴)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- global 로 chat_log 를 수정한다. 정리→호출→누적→트리밍→반환 순.

세부구현:
1. def ask(user_text): 안에서 global chat_log 를 선언한다.
2. 입력을 clean_step.invoke(...) 로 정리해 변수에 담는다.
3. recall 로 꺼낸 사실들을 한 덩어리 글로 만들고, counsel_chain 을 invoke 하되 memory·history·input 을 함께 넘긴다.
4. add_turn 으로 (정리된 입력, 답) 을 누적한 뒤 keep_recent 로 최근 10턴만 남겨 chat_log 에 다시 담는다.
5. 답을 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert callable(ask)
assert chat_log == []   # 아직 부르기 전
print('✅ 통과!')

### 3단계 — 첫 질문 던지기
- `ask` 에 질문 **"무선  이어폰   중고로 사려는데 뭘 확인해야 해?"** 을 넣은 답을 변수 **`ans_a1`** 에 담으세요(질문을 그대로 쓰세요 — 공백 포함).

**예시**: `ans_a1` 은 비어 있지 않은 문자열이고, 호출 후 `chat_log` 길이는 2입니다.

<details><summary>힌트</summary>

```text
접근방법:
- ask(질문) 결과를 변수에 담는다.

세부구현:
1. ans_a1 = ask(질문).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(ans_a1, str) and len(ans_a1.strip()) > 0
assert len(chat_log) == 2   # 한 턴 = 두 줄
print('✅ 통과!')

### 4단계 — 이어 묻기(기억 확인)
- 이어서 `ask` 에 질문 **"그거 배터리는 오래가?"** 을 넣은 답을 변수 **`ans_a2`** 에 담으세요.
- 이 질문의 '그거'는 앞서 말한 무선 이어폰을 가리킵니다 — 봇이 앞 대화를 **기억**하는지 답을 눈으로 확인하세요.

**예시**: `ans_a2` 는 비어 있지 않은 문자열이고, 두 턴이 쌓여 `chat_log` 길이는 4입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3단계와 같은 방식으로 이어서 부른다.

세부구현:
1. ans_a2 = ask(질문).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(ans_a2, str) and len(ans_a2.strip()) > 0
assert len(chat_log) == 4   # 두 턴 = 네 줄 (상한 10턴 안이라 아직 밀려난 것이 없다)
print('✅ 통과!')

### 5단계 — 장기 기억 확인
- `ask` 에 질문 **"제 조건에 맞는 매물인지 봐 주세요."** 을 넣은 답을 변수 **`ans_a3`** 에 담으세요.
- 이 질문에 제대로 답하려면 **예산과 직거래 위치**를 알아야 하는데, 그 사실은 **오늘 대화에 나온 적이 없습니다** — 제공 셀이 장기 기억에 넣어 둔 것뿐입니다.
- 답을 출력하기 전에, `recall` 이 무엇을 꺼내 왔는지도 함께 출력해 눈으로 확인하세요.

**예시**: `ans_a3` 는 비어 있지 않은 문자열이고, 세 턴이 쌓여 `chat_log` 길이는 6입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3·4단계와 같은 방식이되, 회상 결과를 먼저 출력해 본다.

세부구현:
1. recall(질문) 결과를 출력한다.
2. ans_a3 = ask(질문).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(ans_a3, str) and len(ans_a3.strip()) > 0
assert len(chat_log) == 6   # 세 턴 = 여섯 줄
assert '이 손님의 예산은 5만원 이하이고 정품 부속이 갖춰진 매물만 원한다.' in recall('제 조건에 맞는 매물인지 봐 주세요.')   # 예산 사실이 회상 대상에 들어오는지
print('✅ 통과!')

### 6단계 — `while` 로 멀티턴 돌리기
- 3·4단계는 셀을 하나씩 실행해 한 턴씩 돌렸습니다. 이번엔 **한 셀에서 여러 턴**이 이어지게 만듭니다.
- 아래 **제공된 대기열** `queued` 를 `while` 로 비우세요. 한 바퀴가 한 턴입니다.
  1. 대기열 맨 앞의 말을 꺼낸다(`pop(0)`).
  2. 그 말이 `'그만'` 이면 `break` 로 루프를 빠져나온다.
  3. 아니면 `ask` 로 답을 받아 `손님`·`상담원` 을 한 줄씩 출력한다.
- 루프가 끝난 뒤 `chat_log` 의 줄 수를 변수 **`log_len`** 에 담으세요.

**예시**: 대기열에 질문 2개 + `'그만'` 이 들어 있으므로, 앞 4단계까지의 2턴에 2턴이 더해져 `log_len` 은 10입니다(5턴 x 2줄, 상한 10턴 안).

<details><summary>힌트</summary>

```text
접근방법:
- while 로 대기열을 비우되, 끝내는 말이 나오면 break 한다.

세부구현:
1. while queued: 로 돈다.
2. user_text = queued.pop(0) 로 맨 앞의 말을 꺼낸다.
3. user_text 가 '그만' 이면 break.
4. 아니면 ask(user_text) 로 답을 받아 출력한다.
5. 루프가 끝난 뒤 log_len = len(chat_log).
```

</details>

In [ ]:
# [제공 코드] 손님이 칠 말을 미리 담아 둔 대기열 — 마지막 '그만' 이 대화를 끝내는 말입니다.
queued = [
    '충전 케이블도 같이 주시나요?',
    '직거래 가능한 지역이 어디예요?',
    '그만',
]

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert queued == []   # 대기열을 끝까지 비웠는지('그만'까지 꺼냈어야 한다)
assert log_len == 10   # 5단계까지 3턴 + 이번 2턴 = 5턴 x 2줄
assert chat_log[-1][0] == 'assistant'   # 기록은 언제나 모델 답으로 끝난다
print('✅ 통과!')

---
## 2. 상품 설명 파이프라인
**배경**: 매물 정보(이름·특징)를 받아 **1) 홍보 문구와 2) 상태 요약을 동시에** 만들고, 이를 **합쳐 최종 설명**으로 다듬은 뒤, **여러 매물을 한꺼번에** 처리해 **파일로 저장**하는 파이프라인을 만듭니다.

> 4개 단계: (1) 병렬 생성 → (2) 합성 체인 → (3) 일괄 처리(batch) → (4) 결과 저장.

먼저 일괄 처리에 쓸 **매물 데이터**를 파일에서 읽어 옵니다(아래 제공 셀 실행).

> 이 파이프라인의 프롬프트는 **코드가 아니라 파일**에 있습니다(교안 3절). 아래 제공 셀이 `data/prompts_used.yml` 을 읽어 `PROMPTS` 에 담아 둡니다 — 여러분은 **이름으로 꺼내** 쓰면 됩니다.

In [ ]:
# [제공 코드] 파이프라인이 쓸 프롬프트를 파일에서 읽어 둡니다 — 교안 3절과 같은 구조입니다.
import yaml

with open('data/prompts_used.yml', encoding='utf-8') as f:
    PROMPTS = yaml.safe_load(f)

print('파일에 든 프롬프트:', list(PROMPTS))
for key in PROMPTS:
    print(' -', key, ':', PROMPTS[key]['description'])

In [ ]:
# [제공 코드] 일괄 처리에 쓸 중고 매물 데이터를 CSV 에서 읽어 옵니다(지난 단원 read_csv 복습).
import pandas as pd

devices_df = pd.read_csv('data/used_devices.csv')
display(devices_df.head())

# 이 중 3건을 3단계에서 파이프라인으로 일괄 처리합니다(name·note 열만 사용).
#   isin 으로 고른 이유: 어느 환경에서 실행해도 같은 3건이라 결과를 서로 비교할 수 있습니다.
picked = devices_df[devices_df['device_id'].isin(['d1', 'd4', 'd7'])]
items3 = [{'name': r['name'], 'note': r['note']} for _, r in picked.iterrows()]
print('일괄 처리 대상:', [it['name'] for it in items3])

### 1단계 — 병렬 생성 부품
- 먼저 **이름으로 프롬프트를 만드는 함수** `build_prompt(name)` 을 만드세요 — `PROMPTS[name]` 의 `system`·`human` 으로 `ChatPromptTemplate.from_messages` 를 돌려줍니다(교안 3절 `load_prompt` 와 같은 일).
- **홍보 문구 체인** `head_chain`: `build_prompt('headline') | model | parser`.
- **상태 요약 체인** `cond_chain`: `build_prompt('condition') | model | parser`.
- `RunnableParallel` 로 `headline=head_chain`, `condition=cond_chain` 을 묶어 **`gen_parallel`** 을 만드세요.
- `gen_parallel.invoke({'name': '스마트워치', 'note': '화면 잔상 있음, 밴드 마모, 작동 정상'})` 결과를 변수 **`gen1`** 에 담으세요.

**예시**: `gen1` 은 `headline`·`condition` 두 키를 가진 딕셔너리입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 체인을 만들어 RunnableParallel 로 묶는다. 같은 입력이 두 체인에 동시에 들어간다.

세부구현:
1. build_prompt 는 PROMPTS[이름] 의 system·human 을 from_messages 에 넣어 돌려준다.
2. 그 함수로 만든 프롬프트에 model | parser 를 이어 head_chain·cond_chain 을 만든다.
2. gen_parallel = RunnableParallel(headline=head_chain, condition=cond_chain).
3. gen1 = gen_parallel.invoke({'name': ..., 'note': ...}).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# steps__ 는 RunnableParallel 이 들고 있는 '갈래 이름 -> 부품' 묶음이다.
assert set(gen_parallel.steps__) == {'headline', 'condition'}   # 두 갈래를 실제로 묶었는지
# 프롬프트를 코드에 적지 않고 파일에서 읽어 왔는지 — 파일의 글과 같은지 대조한다.
assert head_chain.steps[0].messages[1].prompt.template == PROMPTS['headline']['human']
assert set(gen1.keys()) == {'headline', 'condition'}
assert all(isinstance(v, str) and len(v.strip()) > 0 for v in gen1.values())
print('✅ 통과!')

### 2단계 — 합성 체인
- 병렬 결과의 `headline`·`condition` 을 받아 최종 설명으로 합치는 체인을 만듭니다.
- **합성 프롬프트** `merge_prompt`: 파일의 `merge` 항목을 `build_prompt('merge')` 로 만드세요.
- `gen_parallel` 의 출력(딕셔너리)을 그대로 `merge_prompt | model | parser` 에 넘길 수 있으니, **`pipeline = gen_parallel | merge_prompt | model | parser`** 로 이으세요.
- `pipeline.invoke({'name': '스마트워치', 'note': '화면 잔상 있음, 밴드 마모, 작동 정상'})` 결과를 변수 **`desc2`** 에 담으세요.

**예시**: `desc2` 는 비어 있지 않은 문자열(한 단락)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 병렬 부품의 딕셔너리 출력이 그대로 합성 프롬프트의 변수(headline·condition)로 들어간다.

세부구현:
1. merge_prompt = build_prompt('merge') — 파일의 merge 항목이 이미 {headline}·{condition} 을 받는다.
2. pipeline = gen_parallel | merge_prompt | model | parser.
3. desc2 = pipeline.invoke({'name': ..., 'note': ...}).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(pipeline.steps) == 4   # 병렬 | 합성 프롬프트 | 모델 | 파서
assert isinstance(desc2, str) and len(desc2.strip()) > 0
print('✅ 통과!')

### 3단계 — 여러 매물 일괄 처리(batch)
- 문제 위에서 CSV로 읽어 만든 **`items3`**(매물 3건)를 `pipeline.batch(...)` 로 한꺼번에 처리한 결과 리스트를 변수 **`descs3`** 에 담으세요.

**예시**: `descs3` 는 길이 3의 리스트이고, 각 원소는 문자열입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공된 매물 리스트(items3)를 그대로 pipeline.batch 에 넘긴다.

세부구현:
1. 제공 셀에서 만든 items3 를 pipeline 의 batch 에 넘긴다.
2. 그 결과 리스트를 descs3 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(descs3, list) and len(descs3) == 3
assert all(isinstance(d, str) and len(d.strip()) > 0 for d in descs3)
print('✅ 통과!')

### 4단계 — 결과를 파일로 저장
- 3단계의 매물 이름과 설명을 CSV 로 저장하세요. 경로는 **`output/product_descriptions.csv`** 입니다.
- 첫 줄(헤더)은 `name,description` 이고, 이후 각 줄에 매물 이름과 3단계에서 만든 설명을 적습니다(3줄).
- 저장 경로를 변수 **`out_path`** 에 담으세요.

**예시**: 실행 후 `output/product_descriptions.csv` 파일이 생기고, 데이터 행이 3줄입니다.

<details><summary>힌트</summary>

```text
접근방법:
- output 폴더를 만들고, csv 모듈로 헤더와 각 매물 행을 쓴다.

세부구현:
1. os 모듈의 폴더 생성 함수로 output 폴더를 미리 만들어 둔다(이미 있어도 오류가 나지 않게).
2. csv.writer 로 헤더 ['name','description'] 를 쓴다.
3. items3 의 name 과 descs3 를 짝지어 각 행을 쓴다(zip).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert os.path.exists(out_path)
with open(out_path, encoding='utf-8') as f:
    rows = list(csv.reader(f))
assert rows[0] == ['name', 'description']
assert len(rows) == 4   # 헤더 1줄 + 매물 3줄
print('✅ 통과!')

---
수고했어요! **역할 프롬프트 + 대화 기록 + 입력 정리 + 트리밍**으로 상담 봇을, **병렬 생성 + 합성 + 일괄 처리 + 저장**으로 설명 파이프라인을 완성했습니다. 이 구조가 뒤 단원들에서 더 큰 기능을 쌓는 바탕이 됩니다.